# XGBoost - Cross Validation (FHS)

Notebook para executar XGBoost com StratifiedKFold sobre o dataset FHS.
- Selecione qual dataset (raw ou normalizado) usar alterando a variável `DATASET_PATH` no bloco de configuração.
- Os parâmetros do modelo estão em `xgb_params`.
- Outputs: `model_reports/xgboost/cv/` contém `csv/`, `images/`, `pdf/`.
- O modelo será salvo em `databases/FHS/common/models/xgboost/v1/xgb_model_cf{N_SPLITS}.pkl`. 

In [1]:
# Configuração
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

BASE = Path('../../')  # ajustável se necessário
# Use o dataset raw de FHS por padrão; EDA pode gerar versões normalizadas se desejar
# Nome do dataset cru: fhs_framingham.csv
DATASET_NAME = 'fhs_framingham.csv'  # ou fhs_standard_scaled.csv / fhs_robust_scaled.csv / fhs_minmax_scaled.csv
# Preferir carregar do raw; se quiser usar versões processadas, ajuste a linha abaixo
DATASET_PATH = Path('../data/processed') / DATASET_NAME
# Se quiser forçar explicitamente qual coluna é o target (no bruto ou no normalizado), defina aqui
# Para FHS normalmente a coluna alvo é 'fhs' (0/1)
TARGET_COLUMN = 'TenYearCHD'  # ajustar se necessário

# Output paths
REPORTS = Path('../model_reports/xgboost/cv')
CSV_OUT = REPORTS / 'csv'
IMAGES_OUT = REPORTS / 'images'
PDF_OUT = REPORTS / 'pdf'
MODEL_DIR = Path('../common/models/xgboost/v1')
# Basico outputs (single train/test) - sibling folder to 'cv'
BASICO_DIR = REPORTS.parent / 'basico'
BASICO_CSV = BASICO_DIR / 'csv'
BASICO_IMAGES = BASICO_DIR / 'images'
VARIAVEIS_CATEGORICAS = ['male', 'education', 'currentSmoker', 'BPMeds', 'prevalentStroke', 'prevalentHyp', 'diabetes']

for d in [REPORTS, CSV_OUT, IMAGES_OUT, PDF_OUT, MODEL_DIR, BASICO_DIR, BASICO_CSV, BASICO_IMAGES]:
    d.mkdir(parents=True, exist_ok=True)

# XGBoost params (variável conforme solicitado)
xgb_params = {
    "n_estimators": 650,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.6,
    "gamma": 0.0,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "min_child_weight": 1,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "tree_method": "hist",
    "n_jobs": 8,
    "random_state": 42
}

# CV and threshold config
N_SPLITS = 10
THRESHOLD = 0.5
# Colunas identificadoras que NÃO devem entrar como features no modelo, mas devem ser mantidas nos arquivos de saída
EXCLUDE_COLUMNS = ['id', 'ID', 'patient_id', 'pid','y','stroke','TenYearCHD']

# Pareto configuration: threshold (fraction) and selected-features placeholder (set to None to be updated after CV)
PARETO_THRESHOLD = 0.9  # fraction, e.g. 0.8 == 80%
PARETO_SELECTED_FEATURES = None

# Variance percent-difference threshold (fraction). Example: 0.1 == 10% difference.
VAR_DIFF_PCT_THRESHOLD = 0.01
TEST_SIZE = 0.3
RT_RANDOM_STATE = 42

In [2]:
# Imports principais
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, precision_score, recall_score,
                             balanced_accuracy_score, confusion_matrix, roc_curve, log_loss, auc)
from matplotlib.backends.backend_pdf import PdfPages
from tqdm.auto import tqdm
import joblib

sns.set(style='whitegrid')

In [3]:
# Load dataset (path definido na configuração)
import shutil
if not DATASET_PATH.exists():
    raise FileNotFoundError(f'Arquivo de dataset não encontrado: {DATASET_PATH.resolve()}')

# Preserve a raw copy of the original CSV (unaltered) in data/processed with suffix _raw
RAW_COPY_PATH = Path('../data/processed') / f"{DATASET_NAME.replace('.csv','')}_raw.csv"
try:
    shutil.copy2(DATASET_PATH, RAW_COPY_PATH)
    print(f'Raw dataset copiado para: {RAW_COPY_PATH}')
except Exception as e_copy:
    print('Não foi possível copiar o CSV original para processed:', e_copy)

# Leia o dataset (continuamos a usar df para processamento posterior)
df = pd.read_csv(DATASET_PATH)
# Função utilitária para encontrar coluna aproximada (case/strip tolerant)
def find_column(df, name):
    # tenta match exato
    if name in df.columns:
        return name
    # tenta match case-insensitive
    low = {c.lower().strip(): c for c in df.columns}
    if name.lower().strip() in low:
        return low[name.lower().strip()]
    # tenta encontrar coluna que contenha o nome
    for c in df.columns:
        if name.lower().strip() in c.lower():
            return c
    return None

# Determina a coluna alvo: usa TARGET_COLUMN quando fornecida, senão infere
target_col = None
if TARGET_COLUMN:
    candidate = find_column(df, TARGET_COLUMN)
    if candidate is None:
        raise ValueError(f"TARGET_COLUMN='{TARGET_COLUMN}' não encontrada. Colunas: {list(df.columns)}")
    target_col = candidate
else:
    # heurística leve: nomes comuns
    for cand in ['y', 'TenYearCHD','diagnosis', 'target', 'label', 'class', 'diagnose']:
        candidate = find_column(df, cand)
        if candidate:
            target_col = candidate
            break
    # procura colunas binárias
    if target_col is None:
        for c in df.columns:
            if c.lower().strip() in ('id', 'patient_id', 'pid','TenYearCHD'):
                continue
            try:
                nunq = df[c].nunique(dropna=True)
            except Exception:
                nunq = 999
            if nunq == 2:
                target_col = c
                break
    # fallback para última coluna com poucos valores distintos
    if target_col is None:
        lastc = df.columns[-1]
        if df[lastc].nunique(dropna=True) <= 10:
            target_col = lastc
    if target_col is None:
        raise ValueError(f"Não foi possível inferir a coluna alvo. Defina TARGET_COLUMN ou verifique as colunas: {list(df.columns)}")

# Renomeia para 'y' e garante coluna disponível antes de acessar
if target_col != 'y':
    df = df.rename(columns={target_col: 'y'})
if 'y' not in df.columns:
    raise KeyError('Coluna y não encontrada após renomeação')

# Se y não for numérico, discretiza (M/B -> 1/0 ou factorize para múltiplas classes)
if not np.issubdtype(df['y'].dtype, np.number):
    unique_vals = list(df['y'].dropna().unique())
    low = [str(u).lower() for u in unique_vals]
    if set(low) <= set(['m', 'b']):
        mapping = {v: (1 if str(v).lower() == 'm' else 0) for v in unique_vals}
        df['y'] = df['y'].map(mapping)
    else:
        # factorize para inteiros 0..k-1
        df['y'], uniques = pd.factorize(df['y'])
    # salva versão discretizada para inspeção
    df.to_csv(CSV_OUT / f'database_used_{DATASET_NAME}_with_y_numeric.csv', index=False)

# Substitui inf valores e dropna (pode ajustar imputação se preferir)
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df = df.dropna()

# reorganiza para garantir que y seja a última coluna
cols = [c for c in df.columns if c != 'y'] + ['y']
df = df[cols]

# salva uma cópia de entrada no csv de reports (final)
df.to_csv(CSV_OUT / f'database_used_{DATASET_NAME}', index=False)

print('Dataset shape:', df.shape)

Raw dataset copiado para: ..\data\processed\fhs_framingham_raw.csv
Dataset shape: (3180, 17)


In [5]:
# ==============================================================
# Optuna + XGBoost (compatível com xgboost antigo)
# 8/1/1 hold-out + Optuna (AUC ou ACC) + threshold automático via validação
# Saídas: print + CSV final (val/test) com ACC F1 BAC PRE REC AUC CE-C0 CE-C1 CE-C2 y TP% FP% TN% FN%
# ==============================================================

import os, warnings, json, joblib
import numpy as np
import pandas as pd
from pathlib import Path
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import (
    roc_auc_score, accuracy_score, f1_score, precision_score, recall_score
)
from xgboost import XGBClassifier
import xgboost

warnings.filterwarnings("ignore")

# ========================
# CONFIGS
# ========================
N_TRIALS = 10
RANDOM_STATE = 42
N_SPLITS_INNER = 8
N_SPLITS_OUTER = 10
EPS = 1e-12

# Optuna otimiza o quê (probabilidades => AUC) ou (classes => ACC via threshold fixo interno 0.5)
OPTIMIZE_METRIC = globals().get("OPTIMIZE_METRIC", "auc").lower().strip()
if OPTIMIZE_METRIC not in {"auc", "accuracy"}:
    raise ValueError("OPTIMIZE_METRIC deve ser 'auc' ou 'accuracy'.")

# Threshold automático otimiza o quê (no hold-out de validação)
THRESH_OPTIMIZE_FOR = globals().get("THRESH_OPTIMIZE_FOR", "accuracy").lower().strip()
if THRESH_OPTIMIZE_FOR not in {"accuracy", "f1", "bac"}:
    raise ValueError("THRESH_OPTIMIZE_FOR deve ser 'accuracy', 'f1' ou 'bac'.")

# grade de thresholds (pode refinar: 0.01, 0.005 etc.)
THRESH_GRID_STEP = float(globals().get("THRESH_GRID_STEP", 0.01))

OUT_DIR = Path("../model_reports/xgboost/cv811")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"📦 XGBoost versão detectada: {xgboost.__version__}")
print(f"🎯 Optuna otimizando por: {OPTIMIZE_METRIC.upper()}")
print(f"🎚️ Threshold automático otimiza: {THRESH_OPTIMIZE_FOR.upper()} (usando validação hold-out)")

# ========================
# PREPARO X, y
# ========================
try:
    df
except NameError:
    raise RuntimeError("df não está definido.")

try:
    EXCLUDE_COLUMNS
except NameError:
    EXCLUDE_COLUMNS = [
        '__cls_tmp__', 'orig_index', 'diagnosis', 'id', 'ID',
        'patient_id', 'pid', 'fold', 'y_train', 'prob_0', 'prob_1',
        'y_proba', 'y_pred'
    ]

y = df["y"].copy()

exclude_lower = {c.lower() for c in EXCLUDE_COLUMNS}
ID_COLS = [c for c in df.columns if c.lower() in exclude_lower]
ID_COLS = [c for c in ID_COLS if c != "y"]

FEATURES = [c for c in df.columns if c not in ID_COLS + ["y"]]
X_model = df[FEATURES].copy().replace([np.inf, -np.inf], np.nan)

cat_cols = X_model.select_dtypes(include=["object", "category"]).columns.tolist()
if len(cat_cols) > 0:
    encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X_model[cat_cols] = encoder.fit_transform(X_model[cat_cols])

X_model = X_model.apply(pd.to_numeric, errors='coerce')
X_model = X_model.fillna(X_model.median(numeric_only=True))

# garante binário 0/1
classes = np.unique(y.dropna().values)
if len(classes) != 2:
    raise RuntimeError("Este script está configurado para classificação binária.")
class_to_bin = {classes[0]: 0, classes[1]: 1}
y_bin = y.map(class_to_bin).astype(int)

print(f"✅ X_model pronto: {X_model.shape}, y: {y_bin.shape}, classes: {classes} -> (0/1)")

# ========================
# DIVISÃO 8/1/1 (HOLD-OUT)
# ========================
outer_kf = StratifiedKFold(n_splits=N_SPLITS_OUTER, shuffle=True, random_state=RANDOM_STATE)
fold_indices = list(outer_kf.split(X_model, y_bin))

val_idx  = fold_indices[-2][1]   # fold 9
test_idx = fold_indices[-1][1]   # fold 10

train_mask = np.ones(len(X_model), dtype=bool)
train_mask[val_idx] = False
train_mask[test_idx] = False
train_idx = np.arange(len(X_model))[train_mask]

X_train8, y_train8 = X_model.iloc[train_idx], y_bin.iloc[train_idx]
X_val1,   y_val1   = X_model.iloc[val_idx],   y_bin.iloc[val_idx]
X_test1,  y_test1  = X_model.iloc[test_idx],  y_bin.iloc[test_idx]

print("\n=== Split 8/1/1 (hold-out) ===")
print(f"Treino (8/10): {len(train_idx)} | Validação hold-out (1/10): {len(val_idx)} | Teste hold-out (1/10): {len(test_idx)}")
print("Validação e Teste são hold-out: NÃO entram no Optuna.\n")

# ========================
# FUNÇÕES AUXILIARES
# ========================
def cross_entropy_per_class(y_true, p_pos, eps=1e-12):
    p_pos = np.clip(np.asarray(p_pos), eps, 1 - eps)
    y_true = np.asarray(y_true).astype(int)
    mask0 = (y_true == 0)
    mask1 = (y_true == 1)
    ce0 = float(np.mean(-np.log(1 - p_pos[mask0]))) if np.any(mask0) else np.nan
    ce1 = float(np.mean(-np.log(p_pos[mask1])))     if np.any(mask1) else np.nan
    return ce0, ce1

def safe_auc(y_true, p_pos):
    y_true = np.asarray(y_true).astype(int)
    if len(np.unique(y_true)) < 2:
        return np.nan
    return float(roc_auc_score(y_true, p_pos))

def bac_from_counts(TP, TN, FP, FN):
    rec1 = TP / (TP + FN) if (TP + FN) > 0 else np.nan
    rec0 = TN / (TN + FP) if (TN + FP) > 0 else np.nan
    return float(np.nanmean([rec0, rec1]))

def metrics_for_table(y_true, p_pos, threshold=0.5, eps=1e-12):
    y_true = np.asarray(y_true).astype(int)
    p_pos  = np.asarray(p_pos)
    y_pred = (p_pos >= threshold).astype(int)

    TP = int(np.sum((y_true == 1) & (y_pred == 1)))
    TN = int(np.sum((y_true == 0) & (y_pred == 0)))
    FP = int(np.sum((y_true == 0) & (y_pred == 1)))
    FN = int(np.sum((y_true == 1) & (y_pred == 0)))
    total = TP + TN + FP + FN

    acc = float(accuracy_score(y_true, y_pred))
    f1  = float(f1_score(y_true, y_pred, zero_division=0))
    pre = float(precision_score(y_true, y_pred, zero_division=0))
    rec = float(recall_score(y_true, y_pred, zero_division=0))
    bac = bac_from_counts(TP, TN, FP, FN)
    auc = safe_auc(y_true, p_pos)

    ce0, ce1 = cross_entropy_per_class(y_true, p_pos, eps)

    TPp = TP / total if total > 0 else np.nan
    FPp = FP / total if total > 0 else np.nan
    TNp = TN / total if total > 0 else np.nan
    FNp = FN / total if total > 0 else np.nan

    return {
        "ACC": acc, "F1": f1, "BAC": bac, "PRE": pre, "REC": rec, "AUC": auc,
        "CE-C0": ce0, "CE-C1": ce1, "CE-C2": "---", "y": 1,
        "TP%": TPp, "FP%": FPp, "TN%": TNp, "FN%": FNp
    }

def all_metrics_table_legacy(y_true, p_pos, threshold=0.5, eps=1e-12):
    tab = metrics_for_table(y_true, p_pos, threshold, eps)
    return {
        "Acurácia": tab["ACC"],
        "F1 Score": tab["F1"],
        "Balanced Accuracy": tab["BAC"],
        "Precision": tab["PRE"],
        "Recall": tab["REC"],
        "AUROC": tab["AUC"],
        "Cross-Entropy C0": tab["CE-C0"],
        "Cross-Entropy C1": tab["CE-C1"],
        "TP(%)": tab["TP%"],
        "FP(%)": tab["FP%"],
        "TN(%)": tab["TN%"],
        "FN(%)": tab["FN%"],
    }

def find_best_threshold(y_true, p_pos, optimize_for="accuracy", step=0.01):
    """
    Escolhe threshold que maximiza ACC/F1/BAC no conjunto (aqui: validação hold-out).
    Importante: NÃO "otimiza AUC" porque AUC independe de threshold.
    """
    y_true = np.asarray(y_true).astype(int)
    p_pos  = np.asarray(p_pos)

    best_t = 0.5
    best_s = -np.inf

    thresholds = np.arange(0.0, 1.0 + 1e-9, step)
    for t in thresholds:
        y_pred = (p_pos >= t).astype(int)

        TP = int(np.sum((y_true == 1) & (y_pred == 1)))
        TN = int(np.sum((y_true == 0) & (y_pred == 0)))
        FP = int(np.sum((y_true == 0) & (y_pred == 1)))
        FN = int(np.sum((y_true == 1) & (y_pred == 0)))

        if optimize_for == "accuracy":
            s = float(accuracy_score(y_true, y_pred))
        elif optimize_for == "f1":
            s = float(f1_score(y_true, y_pred, zero_division=0))
        else:  # "bac"
            s = bac_from_counts(TP, TN, FP, FN)

        if s > best_s:
            best_s = s
            best_t = float(t)

    return best_t, best_s

# ========================
# OBJETIVO OPTUNA (CV INTERNA 8-FOLDS)
# ========================
inner_kf = StratifiedKFold(n_splits=N_SPLITS_INNER, shuffle=True, random_state=RANDOM_STATE)

def build_params(trial):
    return dict(
        n_estimators=trial.suggest_int("n_estimators", 200, 800, step=100),
        max_depth=trial.suggest_int("max_depth", 3, 12),
        learning_rate=trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.4, 1.0),
        min_child_weight=trial.suggest_int("min_child_weight", 1, 10),
        gamma=trial.suggest_float("gamma", 0.0, 5.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        tree_method="hist",
        objective="binary:logistic",
        eval_metric="auc",   # mantém compatibilidade
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbosity=0,
    )

def objective(trial):
    params = build_params(trial)
    scores = []

    for fold_id, (tr_idx, va_idx) in enumerate(inner_kf.split(X_train8, y_train8), 1):
        X_tr, X_va = X_train8.iloc[tr_idx], X_train8.iloc[va_idx]
        y_tr, y_va = y_train8.iloc[tr_idx], y_train8.iloc[va_idx]

        model = XGBClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)

        p_va = model.predict_proba(X_va)[:, 1]

        if OPTIMIZE_METRIC == "auc":
            s = safe_auc(y_va, p_va)
            if np.isnan(s):
                s = 0.5
        else:
            y_hat = (p_va >= 0.5).astype(int)  # threshold fixo dentro da CV para manter comparabilidade
            s = float(accuracy_score(y_va, y_hat))

        scores.append(s)
        trial.report(float(np.mean(scores)), step=fold_id)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(scores))

# ========================
# EXECUÇÃO DO OPTUNA
# ========================
study = optuna.create_study(
    study_name=f"xgb_optuna_cv811_{OPTIMIZE_METRIC}_thopt_{THRESH_OPTIMIZE_FOR}",
    direction="maximize",
    sampler=TPESampler(seed=RANDOM_STATE, n_startup_trials=8),
    pruner=MedianPruner(n_warmup_steps=2),
)

study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best_params = study.best_params
print(f"\n🏆 Melhor score médio ({OPTIMIZE_METRIC.upper()}): {study.best_value:.6f}")
print("🎯 Melhores hiperparâmetros:\n", json.dumps(best_params, indent=2))

# ========================
# (1) Inner CV detalhado (mantém threshold=0.5)
# ========================
inner_rows = []
for fold_id, (tr_idx, va_idx) in enumerate(inner_kf.split(X_train8, y_train8), 1):
    X_tr, X_va = X_train8.iloc[tr_idx], X_train8.iloc[va_idx]
    y_tr, y_va = y_train8.iloc[tr_idx], y_train8.iloc[va_idx]

    model = XGBClassifier(**best_params)
    model.fit(X_tr, y_tr)
    p_va = model.predict_proba(X_va)[:, 1]

    row = {"etapa": "inner_cv", "fold": fold_id, **all_metrics_table_legacy(y_va, p_va, threshold=0.5, eps=EPS)}
    inner_rows.append(row)

df_inner = pd.DataFrame(inner_rows)
df_inner.to_csv(OUT_DIR / "cv_811_inner_folds.csv", index=False)

# ========================
# (2) Treina em 8 folds e faz validação hold-out (fold 9)
# ========================
clf_val = XGBClassifier(**best_params)
clf_val.fit(X_train8, y_train8)
p_val = clf_val.predict_proba(X_val1)[:, 1]

# 🔥 threshold automático na validação hold-out
best_thr, best_thr_score = find_best_threshold(
    y_true=y_val1, p_pos=p_val,
    optimize_for=THRESH_OPTIMIZE_FOR,
    step=THRESH_GRID_STEP
)

print(f"\n🎚️ Threshold ótimo (validação hold-out) para {THRESH_OPTIMIZE_FOR.upper()}: {best_thr:.3f} (score={best_thr_score:.6f})")

# métricas da validação usando threshold ótimo (e também registra 0.5 se quiser comparar)
df_val = pd.DataFrame([{
    "etapa": "validacao", "fold": 9,
    **all_metrics_table_legacy(y_val1, p_val, threshold=best_thr, eps=EPS)
}])
df_val.to_csv(OUT_DIR / "cv_811_validation.csv", index=False)

# ========================
# (3) Treino final (8+1) e teste hold-out (fold 10) com threshold ótimo
# ========================
X_train9 = pd.concat([X_train8, X_val1])
y_train9 = pd.concat([y_train8, y_val1])

clf_final = XGBClassifier(**best_params)
clf_final.fit(X_train9, y_train9)
p_test = clf_final.predict_proba(X_test1)[:, 1]

df_test = pd.DataFrame([{
    "etapa": "teste", "fold": 10,
    **all_metrics_table_legacy(y_test1, p_test, threshold=best_thr, eps=EPS)
}])
df_test.to_csv(OUT_DIR / "cv_811_test.csv", index=False)

# ========================
# (4) Consolidação long
# ========================
df_all = pd.concat([df_inner, df_val, df_test], ignore_index=True)
df_all.to_csv(OUT_DIR / "cv_811_results_long.csv", index=False)

# ========================
# (5) TABELA FINAL (Validação + Teste) NO FORMATO DA DISSERTAÇÃO (com threshold ótimo)
# ========================
val_tab  = metrics_for_table(y_val1,  p_val,  threshold=best_thr, eps=EPS)
test_tab = metrics_for_table(y_test1, p_test, threshold=best_thr, eps=EPS)

df_final = pd.DataFrame([
    {"split": "validacao", "threshold": best_thr, **val_tab},
    {"split": "teste",     "threshold": best_thr, **test_tab},
])

cols_order = ["split","threshold","ACC","F1","BAC","PRE","REC","AUC","CE-C0","CE-C1","CE-C2","y","TP%","FP%","TN%","FN%"]
df_final = df_final[cols_order]

print("\n=== MÉTRICAS FINAIS (8/1/1) — threshold ajustado na validação hold-out ===")
print(df_final.to_string(index=False))

final_path = OUT_DIR / f"xgb_811_metrics_val_test_opt_{OPTIMIZE_METRIC}_thr_{THRESH_OPTIMIZE_FOR}.csv"
df_final.to_csv(final_path, index=False)

print("\n✅ CSV final (tabela validação/teste) salvo em:", final_path)

# ========================
# (6) Salvamentos extras
# ========================
joblib.dump(study, OUT_DIR / f"optuna_xgb_cv811_{OPTIMIZE_METRIC}_{N_TRIALS}trials.pkl")
with open(OUT_DIR / f"best_xgb_params_cv811_{OPTIMIZE_METRIC}.json", "w") as f:
    json.dump(best_params, f, indent=2)

joblib.dump(clf_final, OUT_DIR / f"xgb_model_cv811_final_opt_{OPTIMIZE_METRIC}.pkl")

print(f"\n✅ Threshold ótimo usado no teste: {best_thr:.3f}")
print(f"✅ Arquivos salvos em: {OUT_DIR.resolve()}")


[I 2026-01-26 06:08:49,498] A new study created in memory with name: xgb_optuna_cv811_auc_thopt_accuracy


📦 XGBoost versão detectada: 2.1.3
🎯 Optuna otimizando por: AUC
🎚️ Threshold automático otimiza: ACCURACY (usando validação hold-out)
✅ X_model pronto: (3180, 15), y: (3180,), classes: [0 1] -> (0/1)

=== Split 8/1/1 (hold-out) ===
Treino (8/10): 2544 | Validação hold-out (1/10): 318 | Teste hold-out (1/10): 318
Validação e Teste são hold-out: NÃO entram no Optuna.



Best trial: 0. Best value: 0.631848:  10%|█         | 1/10 [00:04<00:44,  4.95s/it]

[I 2026-01-26 06:08:54,443] Trial 0 finished with value: 0.631847785051147 and parameters: {'n_estimators': 400, 'max_depth': 12, 'learning_rate': 0.06504856968981275, 'subsample': 0.7993292420985183, 'colsample_bytree': 0.4936111842654619, 'min_child_weight': 2, 'gamma': 0.2904180608409973, 'reg_alpha': 0.6245760287469893, 'reg_lambda': 0.002570603566117598}. Best is trial 0 with value: 0.631847785051147.


Best trial: 0. Best value: 0.631848:  20%|██        | 2/10 [00:09<00:37,  4.63s/it]

...

[I 2026-01-26 06:09:26,959] Trial 7 finished with value: 0.6972965198350689 and parameters: {'n_estimators': 400, 'max_depth': 5, 'learning_rate': 0.022096526145513846, 'subsample': 0.5704621124873813, 'colsample_bytree': 0.8813181884524238, 'min_child_weight': 1, 'gamma': 4.9344346830025865, 'reg_alpha': 0.08916674715636537, 'reg_lambda': 6.143857495033091e-07}. Best is trial 5 with value: 0.7039547828239057.


Best trial: 5. Best value: 0.703955:  90%|█████████ | 9/10 [00:44<00:05,  5.36s/it]

[I 2026-01-26 06:09:33,759] Trial 8 finished with value: 0.6968439933423503 and parameters: {'n_estimators': 800, 'max_depth': 8, 'learning_rate': 0.0057835683328601424, 'subsample': 0.9661451709558936, 'colsample_bytree': 0.7169030705443645, 'min_child_weight': 10, 'gamma': 3.651834476035047, 'reg_alpha': 0.0028615494462317257, 'reg_lambda': 5.358757267475074}. Best is trial 5 with value: 0.7039547828239057.


Best trial: 5. Best value: 0.703955: 100%|██████████| 10/10 [00:46<00:00,  4.67s/it]


[I 2026-01-26 06:09:36,199] Trial 9 finished with value: 0.6982964986788294 and parameters: {'n_estimators': 200, 'max_depth': 10, 'learning_rate': 0.007622214002737811, 'subsample': 0.6807915298528517, 'colsample_bytree': 0.6892853838813437, 'min_child_weight': 10, 'gamma': 3.5891277857959736, 'reg_alpha': 5.902255053402376e-08, 'reg_lambda': 1.3419306507215916e-08}. Best is trial 5 with value: 0.7039547828239057.

🏆 Melhor score médio (AUC): 0.703955
🎯 Melhores hiperparâmetros:
 {
  "n_estimators": 600,
  "max_depth": 6,
  "learning_rate": 0.01942099825171803,
  "subsample": 0.7733551396716398,
  "colsample_bytree": 0.5109126733153162,
  "min_child_weight": 10,
  "gamma": 3.8756641168055728,
  "reg_alpha": 2.854239907497756,
  "reg_lambda": 1.1309571585271483
}

🎚️ Threshold ótimo (validação hold-out) para ACCURACY: 0.410 (score=0.877358)

=== MÉTRICAS FINAIS (8/1/1) — threshold ajustado na validação hold-out ===
    split  threshold      ACC       F1      BAC  PRE      REC      AUC 

In [ ]:
xgb_params 

In [ ]:
# Contiuar daqui em diante 

xgb_params

In [ ]:
# ==============================================================
# XGBoost - Cross Validation em 8/1/1 (com OrdinalEncoder sem vazamento)
# ==============================================================
import os, time, joblib
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    balanced_accuracy_score, roc_auc_score, confusion_matrix,
    roc_curve, log_loss
)
from xgboost import XGBClassifier
from matplotlib.backends.backend_pdf import PdfPages

# ==============================================================
# Configurações gerais
# ==============================================================
# xgb_params = {
#     'n_estimators': 300,
#     'learning_rate': 0.05,
#     'max_depth': 5,
#     'subsample': 0.8,
#     'colsample_bytree': 0.8,
#     'random_state': 42,
#     'n_jobs': -1,
#     'eval_metric': 'logloss'
# }

THRESHOLD = 0.5
N_SPLITS = 10
DATASET_NAME = "FHS_framingham.csv"
CSV_OUT = Path("../model_reports/xgboost/csv")
PDF_OUT = Path("../model_reports/xgboost/pdf")
MODEL_DIR = Path("../model_reports/xgboost/models")

for p in [CSV_OUT, PDF_OUT, MODEL_DIR]:
    os.makedirs(p, exist_ok=True)

# ==============================================================
# Preparação dos dados X, y (a partir de df e EXCLUDE_COLUMNS já existentes)
# ==============================================================
y = df["y"]

# Detecta colunas identificadoras
exclude_lower = {c.lower() for c in EXCLUDE_COLUMNS}
ID_COLS = [c for c in df.columns if c.lower() in exclude_lower]
ID_COLS = [c for c in ID_COLS if c != "y"]

FEATURES = [c for c in df.columns if c not in ID_COLS + ["y"]]
X_full = df[FEATURES].copy().replace([np.inf, -np.inf], np.nan)

# ==============================================================
# Split 8/1/1 (estratificado): 80% treino, 10% validação, 10% teste
# ==============================================================
sss_outer = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, temp_idx = next(sss_outer.split(X_full, y))

X_train_raw, y_train = X_full.iloc[train_idx].copy(), y.iloc[train_idx].copy()
X_temp_raw, y_temp = X_full.iloc[temp_idx].copy(), y.iloc[temp_idx].copy()

# 10% validação e 10% teste (metade do 20% temporário)
sss_temp = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
val_idx_rel, test_idx_rel = next(sss_temp.split(X_temp_raw, y_temp))

X_val_raw, y_val = X_temp_raw.iloc[val_idx_rel].copy(), y_temp.iloc[val_idx_rel].copy()
X_test_raw, y_test = X_temp_raw.iloc[test_idx_rel].copy(), y_temp.iloc[test_idx_rel].copy()

# ==============================================================
# Encoding e imputação SEM vazamento: fit no treino e transform nos demais
# ==============================================================
# Detecta categóricas no treino
cat_cols = X_train_raw.select_dtypes(include=["object", "category"]).columns.tolist()

if len(cat_cols) > 0:
    print(f"🔤 Codificando {len(cat_cols)} variáveis categóricas com OrdinalEncoder: {cat_cols}")
    encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X_train_raw[cat_cols] = encoder.fit_transform(X_train_raw[cat_cols])
    X_val_raw[cat_cols]   = encoder.transform(X_val_raw[cat_cols])
    X_test_raw[cat_cols]  = encoder.transform(X_test_raw[cat_cols])

# Conversão numérica
X_train_raw = X_train_raw.apply(pd.to_numeric, errors='coerce')
X_val_raw   = X_val_raw.apply(pd.to_numeric, errors='coerce')
X_test_raw  = X_test_raw.apply(pd.to_numeric, errors='coerce')

# Imputação por mediana com base no treino
medianas = X_train_raw.median(numeric_only=True)
X_train = X_train_raw.fillna(medianas)
X_val   = X_val_raw.fillna(medianas)
X_test  = X_test_raw.fillna(medianas)

print(f"✅ Split 8/1/1 concluído. Shapes -> Treino: {X_train.shape}, Validação: {X_val.shape}, Teste: {X_test.shape}")

# ==============================================================
# Cross Validation (apenas dentro do TREINO 80%)
# ==============================================================
kf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
resultados, importancias_lista = [], []

start_total = time.time()
with tqdm(total=N_SPLITS, desc="🔄 Folds K-Fold (80% Treino)") as pbar:
    for fold, (tr_idx, va_idx) in enumerate(kf.split(X_train, y_train), 1):
        t0 = time.time()

        X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
        y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]

        model = XGBClassifier(**xgb_params)
        model.fit(X_tr, y_tr)

        # --- Previsões ---
        y_proba_va_all = model.predict_proba(X_va)
        y_proba_tr_all = model.predict_proba(X_tr)
        y_proba_va_C1  = y_proba_va_all[:, 1]
        y_proba_tr_C1  = y_proba_tr_all[:, 1]
        y_pred_va_bin  = (y_proba_va_C1 >= THRESHOLD).astype(int)
        y_pred_tr_bin  = (y_proba_tr_C1 >= THRESHOLD).astype(int)

        # --- Métricas ---
        cm_tr = confusion_matrix(y_tr, y_pred_tr_bin)
        cm_va = confusion_matrix(y_va, y_pred_va_bin)
        tn_tr, fp_tr, fn_tr, tp_tr = cm_tr.ravel()
        tn_va, fp_va, fn_va, tp_va = cm_va.ravel()

        resultados.append({
            'Conjunto': 'Treino', 'Fold': fold,
            'Acurácia': accuracy_score(y_tr, y_pred_tr_bin),
            'F1': f1_score(y_tr, y_pred_tr_bin),
            'Precision': precision_score(y_tr, y_pred_tr_bin),
            'Recall': recall_score(y_tr, y_pred_tr_bin),
            'Balanced Acc': balanced_accuracy_score(y_tr, y_pred_tr_bin),
            'AUROC': roc_auc_score(y_tr, y_proba_tr_C1),
            'TP': tp_tr, 'FP': fp_tr, 'TN': tn_tr, 'FN': fn_tr,
            'Tempo (s)': round(time.time() - t0, 2)
        })

        resultados.append({
            'Conjunto': 'Validação', 'Fold': fold,
            'Acurácia': accuracy_score(y_va, y_pred_va_bin),
            'F1': f1_score(y_va, y_pred_va_bin),
            'Precision': precision_score(y_va, y_pred_va_bin),
            'Recall': recall_score(y_va, y_pred_va_bin),
            'Balanced Acc': balanced_accuracy_score(y_va, y_pred_va_bin),
            'AUROC': roc_auc_score(y_va, y_proba_va_C1),
            'TP': tp_va, 'FP': fp_va, 'TN': tn_va, 'FN': fn_va,
            'Tempo (s)': round(time.time() - t0, 2)
        })

        # --- Importâncias por fold ---
        importancias = model.feature_importances_
        linha_importancia = {'Fold': fold}
        for nome, valor in zip(FEATURES, importancias):
            linha_importancia[nome] = f"{valor*100:.2f}%"
        importancias_lista.append(linha_importancia)

        # Salva modelo do fold (mantido)
        model_path = MODEL_DIR / f"xgb_model_fold{fold}.pkl"
        joblib.dump(model, model_path)

        pbar.update(1)

# ==============================================================
# Treino final em 90% (Treino+Validação) e avaliação no 10% Teste
# ==============================================================
X_90 = pd.concat([X_train, X_val], axis=0)
y_90 = pd.concat([y_train, y_val], axis=0)

final_model = XGBClassifier(**xgb_params)
final_model.fit(X_90, y_90)

y_proba_test_all = final_model.predict_proba(X_test)
y_proba_test_C1  = y_proba_test_all[:, 1]
y_pred_test_bin  = (y_proba_test_C1 >= THRESHOLD).astype(int)

cm_ts = confusion_matrix(y_test, y_pred_test_bin)
tn_ts, fp_ts, fn_ts, tp_ts = cm_ts.ravel()

resultados.append({
    'Conjunto': 'Teste', 'Fold': 0,  # Fold=0 para marcar avaliação final
    'Acurácia': accuracy_score(y_test, y_pred_test_bin),
    'F1': f1_score(y_test, y_pred_test_bin),
    'Precision': precision_score(y_test, y_pred_test_bin),
    'Recall': recall_score(y_test, y_pred_test_bin),
    'Balanced Acc': balanced_accuracy_score(y_test, y_pred_test_bin),
    'AUROC': roc_auc_score(y_test, y_proba_test_C1),
    'TP': tp_ts, 'FP': fp_ts, 'TN': tn_ts, 'FN': fn_ts,
    'Tempo (s)': round(time.time() - start_total, 2)
})

# Salva o modelo final (opcional)
joblib.dump(final_model, MODEL_DIR / "xgb_model_final_90_10.pkl")

# ==============================================================
# Salvamento final (nomes mantidos)
# ==============================================================
df_resultados = pd.DataFrame(resultados)
pd.DataFrame(importancias_lista).to_csv(CSV_OUT / f"xgb_feature_importances_{N_SPLITS}.csv", index=False)
df_resultados.to_csv(CSV_OUT / f"xgb_cv_results_{N_SPLITS}.csv", index=False)

print(f"✅ Resultados salvos em: {CSV_OUT}")
print("📌 Observação: métricas por fold referem-se ao CV dentro dos 80% (Treino). A última linha é o Teste (10%) após treino em 90% (Treino+Validação).")


In [ ]:
# # Diagnóstico rápido para NaN / tipos em train_df e test_df
# for name, ycol in [('train_df', 'y_train'), ('test_df', 'y_test')]:
#     df_block = globals().get(name)
#     if df_block is None:
#         print(f"{name} não existe")
#         continue
#     if df_block.empty:
#         print(f"{name} existe mas está vazio")
#         continue
#     print(f"\n{name}: shape={df_block.shape}")
#     for col in [ycol, 'prob_1', 'y_proba']:
#         if col in df_block.columns:
#             print(f"  {col}: nulls={df_block[col].isnull().sum()}, dtype={df_block[col].dtype}, unique_count={df_block[col].nunique(dropna=True)}")
#     cols_to_show = [c for c in [ycol, 'prob_1', 'y_proba'] if c in df_block.columns]
#     display(df_block[cols_to_show].head(10))

In [ ]:
# ===================== RECONSTRUIR train_df / test_df (8/1/1 compatível) =====================
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.metrics import roc_curve, auc, confusion_matrix

# Pastas auxiliares (mantido)
try:
    IMAGES_OUT
except NameError:
    IMAGES_OUT = PDF_OUT.parent / "images"
IMAGES_OUT.mkdir(parents=True, exist_ok=True)

def _safe_concat(lst):
    return pd.concat(lst, ignore_index=True) if (isinstance(lst, list) and len(lst) > 0) else pd.DataFrame()

def _rename_index_col(_df):
    if isinstance(_df, pd.DataFrame) and ('index' in _df.columns):
        _df = _df.rename(columns={'index': 'orig_index'})
    return _df

def _merge_on_keys(left, right, keys_pref=('orig_index','fold')):
    """Merge com tolerância à ausência de 'fold': usa ['orig_index','fold'] se ambos têm; senão ['orig_index']."""
    keys = []
    for k in keys_pref:
        if k == 'fold':
            if ('fold' in left.columns) and ('fold' in right.columns):
                keys.append('fold')
        else:
            if k in left.columns and k in right.columns:
                keys.append(k)
    if not keys:
        # fallback final: tenta por índice
        left = left.reset_index().rename(columns={'index':'orig_index'}) if 'orig_index' not in left.columns else left
        right = right.reset_index().rename(columns={'index':'orig_index'}) if 'orig_index' not in right.columns else right
        keys = ['orig_index']
    return left.merge(right, on=keys, how='left')

def _ensure_pred_cols(df_block, threshold):
    if 'prob_1' in df_block.columns and 'y_proba' not in df_block.columns:
        df_block['y_proba'] = df_block['prob_1']
    if 'y_proba' in df_block.columns and 'y_pred' not in df_block.columns:
        df_block['y_pred'] = (df_block['y_proba'] >= threshold).astype(int)
    return df_block

# Tenta reconstruir os dataframes globais a partir das partes do CV (80% treino, 10 folds)
X_train_global = _rename_index_col(_safe_concat(globals().get('X_train_parts', [])))
y_train_global = _rename_index_col(_safe_concat(globals().get('y_train_parts', [])))
y_train_pred_global = _rename_index_col(_safe_concat(globals().get('y_train_pred_parts', [])))

X_test_global  = _rename_index_col(_safe_concat(globals().get('X_test_parts', [])))
y_test_global  = _rename_index_col(_safe_concat(globals().get('y_test_parts', [])))
y_test_pred_global  = _rename_index_col(_safe_concat(globals().get('y_test_pred_parts', [])))

# Monta train_df (OOF dos 80%)
if not X_train_global.empty and not y_train_global.empty:
    train_df = _merge_on_keys(X_train_global, y_train_global)
    if not y_train_pred_global.empty:
        train_df = _merge_on_keys(train_df, y_train_pred_global)
    train_df = _ensure_pred_cols(train_df, THRESHOLD)
else:
    train_df = pd.DataFrame()

# Monta test_df:
# 1) preferir partes globais (se você armazenou fold=0 ou partes de teste);
# 2) senão, usar X_test / y_test e previsões do final_model (fold=0).
if not X_test_global.empty and not y_test_global.empty:
    test_df = _merge_on_keys(X_test_global, y_test_global)
    if not y_test_pred_global.empty:
        test_df = _merge_on_keys(test_df, y_test_pred_global)
    test_df = _ensure_pred_cols(test_df, THRESHOLD)

    # Se não houver 'fold', marcar teste final com fold=0
    if 'fold' not in test_df.columns:
        test_df['fold'] = 0
else:
    # Fallback 8/1/1: usar variáveis do split final (10% teste)
    # Espera-se que X_test, y_test e final_model existam do pipeline 8/1/1
    if 'X_test' in globals() and 'y_test' in globals() and 'final_model' in globals():
        _xt = X_test.copy()
        # garantir chave de junção
        if 'orig_index' not in _xt.columns:
            _xt = _xt.reset_index().rename(columns={'index':'orig_index'})
        test_df = _xt
        test_df['y_test'] = y_test.values
        # prever com o modelo final 90/10
        try:
            _proba = final_model.predict_proba(X_test)[:, 1]
            test_df['prob_1'] = _proba
            test_df['fold'] = 0  # marca teste final
            test_df = _ensure_pred_cols(test_df, THRESHOLD)
        except Exception as _e:
            # sem proba: segue sem ROC
            pass
    else:
        test_df = pd.DataFrame()

# ===================== PDF RESUMO XGBOOST (nomes mantidos) =====================
with PdfPages(PDF_OUT / f'XGB_CV_SUMMARY_{N_SPLITS}.pdf') as pdf:
    # -------- Página de parâmetros --------
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.axis('off')
    parametros = xgb_params.copy()
    parametros['threshold'] = THRESHOLD
    parametros['n_splits'] = N_SPLITS
    texto = 'Algoritmo: XGBoost\n' + '\n'.join([f'{k}: {v}' for k, v in parametros.items()])
    ax.text(0, 1, texto, fontsize=11, family='monospace', va='top')
    ax.set_title('📌 Parâmetros do Modelo - XGBoost')
    try:
        png_path = IMAGES_OUT / f'params_{N_SPLITS}.png'
        fig.savefig(png_path, dpi=300, bbox_inches='tight')
    except Exception:
        pass
    pdf.savefig(fig); plt.close()

    # -------- Página de métricas resumidas --------
    if 'df_resultados' in globals() and not df_resultados.empty:
        fig, ax = plt.subplots(figsize=(12, 6))
        ax.axis('off')
        display_df = df_resultados.groupby(['Conjunto']).mean(numeric_only=True).T.round(4)
        table = ax.table(cellText=display_df.values,
                         colLabels=display_df.columns,
                         rowLabels=display_df.index,
                         loc='center')
        table.auto_set_font_size(False); table.set_fontsize(9)
        try:
            png_path = IMAGES_OUT / f'metrics_summary_{N_SPLITS}.png'
            fig.savefig(png_path, dpi=300, bbox_inches='tight')
        except Exception:
            pass
        pdf.savefig(fig); plt.close()

    # -------- ROC - Treino (OOF nos 80%) vs Teste (10% final) --------
    try:
        plotted = False
        # Treino (OOF): se reconstruído
        if ('y_train' in train_df.columns) and ('y_proba' in train_df.columns):
            y_train_all = train_df['y_train']; y_score_train_all = train_df['y_proba']
            fpr_train, tpr_train, _ = roc_curve(y_train_all, y_score_train_all)
            roc_auc_train = auc(fpr_train, tpr_train); plotted = True
        elif 'fprs_train' in globals() and len(fprs_train) > 0:
            mean_fpr = np.linspace(0, 1, 200)
            tprs = [np.interp(mean_fpr, fpr, tpr) for fpr, tpr in zip(fprs_train, tprs_train)]
            for t in tprs: t[0] = 0.0
            mean_tpr = np.mean(tprs, axis=0); mean_tpr[-1] = 1.0
            fpr_train, tpr_train = mean_fpr, mean_tpr
            roc_auc_train = np.mean(aurocs_train) if 'aurocs_train' in globals() and len(aurocs_train)>0 else auc(fpr_train,tpr_train)
            plotted = True

        # Teste (10% final)
        if ('y_test' in test_df.columns) and ('y_proba' in test_df.columns):
            y_test_all = test_df['y_test']; y_score_test_all = test_df['y_proba']
            fpr_test, tpr_test, _ = roc_curve(y_test_all, y_score_test_all)
            roc_auc_test = auc(fpr_test, tpr_test); plotted = True
        elif 'fprs_test' in globals() and len(fprs_test) > 0:
            mean_fpr = np.linspace(0, 1, 200)
            tprs = [np.interp(mean_fpr, fpr, tpr) for fpr, tpr in zip(fprs_test, tprs_test)]
            for t in tprs: t[0] = 0.0
            mean_tpr = np.mean(tprs, axis=0); mean_tpr[-1] = 1.0
            fpr_test, tpr_test = mean_fpr, mean_tpr
            roc_auc_test = np.mean(aurocs_test) if 'aurocs_test' in globals() and len(aurocs_test)>0 else auc(fpr_test,tpr_test)
            plotted = True

        if plotted:
            fig, axes = plt.subplots(1, 2, figsize=(12, 5))
            try:
                axes[0].plot(fpr_train, tpr_train, label=f'AUC = {roc_auc_train:.4f}')
                axes[0].plot([0,1],[0,1],'k--',lw=1)
                axes[0].set_title('ROC - Treino (80% OOF)'); axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
                axes[0].legend(loc='lower right')
            except Exception:
                axes[0].text(0.5,0.5,'Sem dados de treino',ha='center'); axes[0].axis('off')
            try:
                axes[1].plot(fpr_test, tpr_test, label=f'AUC = {roc_auc_test:.4f}')
                axes[1].plot([0,1],[0,1],'k--',lw=1)
                axes[1].set_title('ROC - Teste (10%)'); axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
                axes[1].legend(loc='lower right')
            except Exception:
                axes[1].text(0.5,0.5,'Sem dados de teste',ha='center'); axes[1].axis('off')
            plt.tight_layout()
            try:
                png_path = IMAGES_OUT / f'roc_train_test_{N_SPLITS}.png'
                fig.savefig(png_path, dpi=300, bbox_inches='tight')
            except Exception:
                pass
            pdf.savefig(fig, bbox_inches='tight'); plt.close()
        else:
            print('Não foi possível gerar curvas ROC (dados ausentes)')
    except Exception as e:
        print('Não foi possível gerar ROC agregado:', e)

    # -------- Matrizes de Confusão (80% OOF vs 10% final) --------
    try:
        def build_cm_from_df(df_block, true_col, pred_col):
            y_true = df_block[true_col]; y_pred = df_block[pred_col]
            cm = confusion_matrix(y_true, y_pred)
            total = cm.sum(); cm_percent = cm/total*100 if total>0 else np.zeros_like(cm, dtype=float)
            return cm, cm_percent

        cm_train = cm_train_pct = cm_test = cm_test_pct = None

        if ('y_train' in train_df.columns) and ('y_pred' in train_df.columns):
            cm_train, cm_train_pct = build_cm_from_df(train_df, 'y_train', 'y_pred')
        elif ('df_resultados' in globals()) and ('TP' in df_resultados.columns):
            tr = df_resultados[df_resultados['Conjunto']=='Treino']
            TP,FP,TN,FN = int(tr['TP'].sum()), int(tr['FP'].sum()), int(tr['TN'].sum()), int(tr['FN'].sum())
            cm_train = np.array([[TN, FP],[FN, TP]])
            total = cm_train.sum(); cm_train_pct = cm_train/total*100 if total>0 else np.zeros_like(cm_train,float)

        if ('y_test' in test_df.columns) and ('y_pred' in test_df.columns):
            cm_test, cm_test_pct = build_cm_from_df(test_df, 'y_test', 'y_pred')
        elif ('df_resultados' in globals()) and ('TP' in df_resultados.columns):
            ts = df_resultados[df_resultados['Conjunto']=='Teste']
            TP,FP,TN,FN = int(ts['TP'].sum()), int(ts['FP'].sum()), int(ts['TN'].sum()), int(ts['FN'].sum())
            cm_test = np.array([[TN, FP],[FN, TP]])
            total = cm_test.sum(); cm_test_pct = cm_test/total*100 if total>0 else np.zeros_like(cm_test,float)

        if (cm_train is None and cm_test is None):
            print('Não há dados suficientes para gerar matrizes de confusão.')
        else:
            fig, axes = plt.subplots(1, 2, figsize=(12, 5))

            def plot_cm(ax, cm, cm_pct, title):
                if cm is None:
                    ax.text(0.5,0.5,'Sem dados',ha='center',va='center'); ax.axis('off'); return
                annot = np.array([[f"{int(cm[i,j])}\n({cm_pct[i,j]:.2f}%)" for j in range(cm.shape[1])] for i in range(cm.shape[0])])
                sns.heatmap(cm, annot=annot, fmt='', cmap='Blues', ax=ax, cbar=False, annot_kws={'size':12})
                ax.set_xlabel('Predito'); ax.set_ylabel('Verdadeiro')
                ax.set_xticklabels(['0','1']); ax.set_yticklabels(['0','1'])
                ax.set_title(title)

            plot_cm(axes[0], cm_train, cm_train_pct, 'Matriz de Confusão - Treino (80% OOF)')
            plot_cm(axes[1], cm_test,  cm_test_pct,  'Matriz de Confusão - Teste (10%)')

            plt.tight_layout()
            try:
                png_path = IMAGES_OUT / f'confusion_train_test_{N_SPLITS}.png'
                fig.savefig(png_path, dpi=300, bbox_inches='tight')
            except Exception:
                pass
            pdf.savefig(fig, bbox_inches='tight'); plt.close()
    except Exception as e:
        print('Não foi possível gerar matrizes de confusão:', e)

    # -------- Importâncias médias + Pareto (mantido) --------
    try:
        df_imp = pd.DataFrame(importancias_lista).replace('%','', regex=True)
        cols = [c for c in df_imp.columns if c != 'Fold']
        try:
            exclude_lower = {c.lower() for c in EXCLUDE_COLUMNS}
        except Exception:
            exclude_lower = set()
        cols = [c for c in cols if c.lower() not in exclude_lower]

        if len(cols) == 0:
            print('Nenhuma feature disponível para plotar importâncias (após excluir ids).')
        else:
            df_imp_mean = df_imp[cols].astype(float).mean().sort_values(ascending=True)
            height = max(6, 0.25 * len(df_imp_mean))
            fig, ax = plt.subplots(figsize=(10, height))
            df_imp_mean.plot(kind='barh', ax=ax, color='tab:blue')
            ax.set_xlabel('Importância média (%)'); ax.set_title('Importância média das features (%)')
            plt.tight_layout(); plt.subplots_adjust(left=0.30)
            try:
                (IMAGES_OUT / f'feature_importances_{N_SPLITS}.png').write_bytes(b'')  # touch
                fig.savefig(IMAGES_OUT / f'feature_importances_{N_SPLITS}.png', dpi=300, bbox_inches='tight')
            except Exception:
                pass
            pdf.savefig(fig, bbox_inches='tight'); plt.close()

            # Tabela
            try:
                table_df = df_imp_mean.sort_values(ascending=False).to_frame(name='Importance (%)').round(2)
                fig, ax = plt.subplots(figsize=(8, max(4, 0.3*len(table_df))))
                ax.axis('off')
                ax.table(cellText=table_df.values, colLabels=table_df.columns,
                         rowLabels=table_df.index, loc='center')
                ax.set_title('Tabela: Importância média das features (%)', pad=20)
                plt.tight_layout(); pdf.savefig(fig, bbox_inches='tight'); plt.close()
            except Exception as e_table:
                print('Não foi possível gerar tabela de importâncias:', e_table)

            # Pareto 80/20
            try:
                imp_desc = df_imp_mean.sort_values(ascending=False)
                pareto_df = imp_desc.to_frame(name='importance_pct')
                pareto_df['cumulative_pct'] = pareto_df['importance_pct'].cumsum()
                threshold_pct = (PARETO_THRESHOLD if 'PARETO_THRESHOLD' in globals() else 0.8)*100
                import numpy as _np
                pos = _np.searchsorted(pareto_df['cumulative_pct'].values, threshold_pct)
                selected_features = pareto_df.iloc[:pos+1].index.tolist() if pos < len(pareto_df) else pareto_df.index.tolist()
                PARETO_SELECTED_FEATURES = selected_features

                # Gráfico Pareto
                fig, ax1 = plt.subplots(figsize=(10, 6))
                x = pareto_df.index.astype(str)
                ax1.bar(x, pareto_df['importance_pct'], color='tab:blue')
                ax1.set_ylabel('Importância (%)'); ax1.set_xticklabels(x, rotation=90)
                ax2 = ax1.twinx(); ax2.plot(x, pareto_df['cumulative_pct'], color='red', marker='o')
                ax2.axhline(threshold_pct, color='gray', linestyle='--'); ax2.set_ylabel('Cumulative (%)')
                ax1.set_title(f'Pareto (threshold={int(threshold_pct)}%) - selecionadas: {len(PARETO_SELECTED_FEATURES)}')
                plt.tight_layout()
                try:
                    fig.savefig(IMAGES_OUT / f'pareto_{int(threshold_pct)}.png', dpi=300, bbox_inches='tight')
                except Exception:
                    pass
                pdf.savefig(fig, bbox_inches='tight'); plt.close()
            except Exception as e_pareto:
                print('Não foi possível gerar análise Pareto:', e_pareto)
    except Exception as e:
        print('Não foi possível gerar gráfico de importâncias:', e)

    # -------- Tabela de variâncias (numéricas) Treino OOF vs Teste 10% --------
    try:
        if not (train_df.empty or test_df.empty):
            feats_common = [c for c in train_df.columns if (c in test_df.columns)]
            drop_meta = set(['fold','orig_index','y_train','y_test','y_pred','y_proba','prob_0','prob_1'])
            feats_common = [c for c in feats_common if c not in drop_meta]
            train_num = train_df[feats_common].apply(pd.to_numeric, errors='coerce')
            test_num  = test_df[feats_common].apply(pd.to_numeric, errors='coerce')
            valid_cols = [c for c in feats_common if train_num[c].notna().any() and test_num[c].notna().any()]
            if len(valid_cols) > 0:
                var_train = train_num[valid_cols].var(ddof=1, numeric_only=True)
                var_test  = test_num[valid_cols].var(ddof=1, numeric_only=True)
                var_df = pd.DataFrame({'var_train': var_train, 'var_test': var_test})
                var_df['diff'] = var_df['var_train'] - var_df['var_test']
                var_df['pct_diff'] = (var_df['diff'].abs() / var_df['var_train'].replace({0: np.nan})) * 100
                try:
                    thresh_pct = VAR_DIFF_PCT_THRESHOLD * 100
                except Exception:
                    thresh_pct = 10.0
                var_df['high_low'] = var_df['pct_diff'].apply(lambda x: 'High' if pd.notna(x) and x >= thresh_pct else 'Low')
                var_df = var_df.sort_values('var_train', ascending=False)

                out_var_csv = CSV_OUT / 'feature_variances_train_test.csv'
                var_df.reset_index().rename(columns={'index':'feature'}).to_csv(out_var_csv, index=False)

                fig, ax = plt.subplots(figsize=(11.69, min(12, 0.25*len(var_df)+2)))
                ax.axis('off')
                ax.table(cellText=var_df.round(4).reset_index().values,
                         colLabels=['feature','var_train','var_test','diff','pct_diff','high_low'],
                         loc='center').auto_set_font_size(False)
                ax.set_title('Tabela: Variância das Features (Treino OOF vs Teste 10%)')
                plt.tight_layout(); pdf.savefig(fig, bbox_inches='tight'); plt.close()
            else:
                print('Nenhuma feature numérica válida em comum para variância.')
        else:
            print('Dados de treino/teste ausentes; não foi possível calcular variâncias.')
    except Exception as e_var:
        print('Erro ao calcular tabela de variâncias:', e_var)

    # -------- Violinos para features do Pareto --------
    try:
        if ('PARETO_SELECTED_FEATURES' in globals()) and PARETO_SELECTED_FEATURES and not (train_df.empty or test_df.empty):
            violin_features = [f for f in PARETO_SELECTED_FEATURES if (f in train_df.columns and f in test_df.columns)]
            if len(violin_features) > 0:
                combined = pd.concat([
                    train_df[violin_features].assign(__set__='train'),
                    test_df[violin_features].assign(__set__='test')
                ], ignore_index=True)
                per_page = 6
                import math
                pages = math.ceil(len(violin_features)/per_page)
                for p in range(pages):
                    subset = violin_features[p*per_page : (p+1)*per_page]
                    rows = 3
                    fig, axes = plt.subplots(rows, 2, figsize=(8.27, 11.69))
                    axes = axes.flatten()
                    for i, feat in enumerate(subset):
                        ax = axes[i]
                        sns.violinplot(x='__set__', y=feat, data=combined,
                                       order=['train','test'], ax=ax, inner='quartile',
                                       palette=['#4c72b0','#55a868'])
                        ax.set_title(f'Violin: {feat} (train vs test)')
                        ax.set_xlabel('')
                    for j in range(len(subset), len(axes)):
                        axes[j].axis('off')
                    plt.tight_layout(); pdf.savefig(fig, bbox_inches='tight'); plt.close()
            else:
                print('Nenhuma das features do Pareto está presente em train/test.')
        else:
            print('Sem PARETO_SELECTED_FEATURES ou train/test vazios; violinos omitidos.')
    except Exception as e_violin:
        print('Não foi possível gerar gráficos de violino:', e_violin)

print('PDF resumo gerado em:', PDF_OUT)


In [ ]:
# === XGBoost Basico (agora em 8/1/1, mantendo nomes e salvamentos) ===
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
)
from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBClassifier

# Parâmetros de split (defaults se não existirem)
try:
    TEST_SIZE
except NameError:
    TEST_SIZE = 0.1  # usamos 10% para teste no esquema 8/1/1 (nomes de arquivos continuam usando TEST_SIZE)
try:
    RT_RANDOM_STATE
except NameError:
    RT_RANDOM_STATE = 42

# === Verificações básicas ===
if 'FEATURES' not in globals():
    raise RuntimeError('❌ FEATURES não encontrado — execute as células anteriores para preparar os dados')
if 'df' not in globals():
    raise RuntimeError('❌ DataFrame df não encontrado — execute as células anteriores para carregar os dados')
if 'xgb_params' not in globals():
    raise RuntimeError('❌ xgb_params não encontrado — defina os hiperparâmetros antes de rodar')

# === Cópia base ===
X_all = df[FEATURES].copy().replace([np.inf, -np.inf], np.nan)
y_all = df['y'].copy()

# === Split 8/1/1 (80% treino, 10% validação, 10% teste) ===
sss_outer = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=RT_RANDOM_STATE)
train_idx, temp_idx = next(sss_outer.split(X_all, y_all))

X_train_raw, y_train = X_all.iloc[train_idx].copy(), y_all.iloc[train_idx].copy()
X_temp_raw,  y_temp  = X_all.iloc[temp_idx].copy(),  y_all.iloc[temp_idx].copy()

sss_temp = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=RT_RANDOM_STATE)
val_rel_idx, test_rel_idx = next(sss_temp.split(X_temp_raw, y_temp))

X_val_raw,  y_val  = X_temp_raw.iloc[val_rel_idx].copy(),  y_temp.iloc[val_rel_idx].copy()
X_test_raw, y_test = X_temp_raw.iloc[test_rel_idx].copy(), y_temp.iloc[test_rel_idx].copy()

# === Codificação categórica SEM vazamento (fit no treino, transform nos demais) ===
cat_cols = X_train_raw.select_dtypes(include=['object', 'category']).columns.tolist()
if len(cat_cols) > 0:
    print(f"🔤 Codificando variáveis categóricas com OrdinalEncoder (fit no treino): {cat_cols}")
    encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X_train_raw[cat_cols] = encoder.fit_transform(X_train_raw[cat_cols])
    X_val_raw[cat_cols]   = encoder.transform(X_val_raw[cat_cols])
    X_test_raw[cat_cols]  = encoder.transform(X_test_raw[cat_cols])
else:
    print("✅ Nenhuma variável categórica encontrada para codificação.")

# === Conversão numérica e imputação por mediana (fit no treino) ===
X_train_raw = X_train_raw.apply(pd.to_numeric, errors='coerce')
X_val_raw   = X_val_raw.apply(pd.to_numeric, errors='coerce')
X_test_raw  = X_test_raw.apply(pd.to_numeric, errors='coerce')

medianas = X_train_raw.median(numeric_only=True)
X_train = X_train_raw.fillna(medianas)
X_val   = X_val_raw.fillna(medianas)
X_test  = X_test_raw.fillna(medianas)

print(f"✅ Splits prontos. Treino: {X_train.shape}, Validação: {X_val.shape}, Teste: {X_test.shape}")

# === Treina modelo final em 90% (treino+validação) ===
X_90 = pd.concat([X_train, X_val], axis=0)
y_90 = pd.concat([y_train, y_val], axis=0)

model_basic = XGBClassifier(**xgb_params)
model_basic.fit(X_90, y_90)

# === Predições ===
y_pred_train_bin = model_basic.predict(X_90)
y_pred_test_bin  = model_basic.predict(X_test)
y_proba_train = model_basic.predict_proba(X_90)[:, 1]
y_proba_test  = model_basic.predict_proba(X_test)[:, 1]

# === Métricas (treino aqui é 90%) ===
acc_train = accuracy_score(y_90, y_pred_train_bin)
acc_test  = accuracy_score(y_test, y_pred_test_bin)
f1_tr = f1_score(y_90, y_pred_train_bin)
f1_ts = f1_score(y_test, y_pred_test_bin)
prec_tr = precision_score(y_90, y_pred_train_bin)
prec_ts = precision_score(y_test, y_pred_test_bin)
rec_tr = recall_score(y_90, y_pred_train_bin)
rec_ts = recall_score(y_test, y_pred_test_bin)
auc_tr = roc_auc_score(y_90, y_proba_train)
auc_ts = roc_auc_score(y_test, y_proba_test)

print(f"✅ Acurácia treino(90%): {acc_train:.4f} — teste(10%): {acc_test:.4f}")

# === Salvamentos (mantidos) ===
model_name = MODEL_DIR / f'xgb_basic_ts{int(TEST_SIZE*100)}_rs{RT_RANDOM_STATE}.pkl'
joblib.dump(model_basic, model_name)

X_train.to_csv(BASICO_CSV / f'X_train_xgb_basic_ts{int(TEST_SIZE*100)}_rs{RT_RANDOM_STATE}.csv', index=False)
X_test.to_csv( BASICO_CSV / f'X_test_xgb_basic_ts{int(TEST_SIZE*100)}_rs{RT_RANDOM_STATE}.csv', index=False)
# y_*: preserva índice para rastreabilidade
y_90.to_csv(   BASICO_CSV / f'y_train_xgb_basic_ts{int(TEST_SIZE*100)}_rs{RT_RANDOM_STATE}.csv', index=True)
y_test.to_csv( BASICO_CSV / f'y_test_xgb_basic_ts{int(TEST_SIZE*100)}_rs{RT_RANDOM_STATE}.csv', index=True)
joblib.dump(FEATURES, MODEL_DIR / f'features_xgb_basic_ts{int(TEST_SIZE*100)}_rs{RT_RANDOM_STATE}.pkl')

metrics_basic = pd.DataFrame([{
    'model': str(model_name.name),
    'test_size': TEST_SIZE,            # segue no CSV para manter compatibilidade
    'random_state': RT_RANDOM_STATE,
    'acc_train': acc_train,            # métricas do fit em 90%
    'acc_test': acc_test,
    'f1_train': f1_tr,
    'f1_test': f1_ts,
    'precision_train': prec_tr,
    'precision_test': prec_ts,
    'recall_train': rec_tr,
    'recall_test': rec_ts,
    'auc_train': auc_tr,
    'auc_test': auc_ts
}])
metrics_basic.to_csv(BASICO_CSV / 'xgb_model_basic.csv', index=False)

# === Dataset augmentado (com previsões no TESTE 10%) ===
df_aug = df.copy()
df_aug['y_pred'] = np.nan
df_aug['y_proba'] = np.nan
df_aug.loc[X_test.index, 'y_pred'] = y_pred_test_bin
df_aug.loc[X_test.index, 'y_proba'] = y_proba_test
df_aug.to_csv(
    BASICO_CSV / f'database_used_{DATASET_NAME}_with_preds_xgb_basic_ts{int(TEST_SIZE*100)}_rs{RT_RANDOM_STATE}.csv',
    index=False
)

# === Amostras aleatórias para explicabilidade (sobre o TESTE 10%) ===
num_instancias = 12
np.random.seed(42)
indices_aleatorios = np.random.choice(X_test.index, size=min(num_instancias, len(X_test)), replace=False)

print('📦 Modelo salvo em:', model_name)
print('📊 Métricas salvas em:', BASICO_CSV / 'xgb_model_basic.csv')
print('🧩 Dataset augmentado salvo em:', BASICO_CSV / f'database_used_{DATASET_NAME}_with_preds_xgb_basic_ts{int(TEST_SIZE*100)}_rs{RT_RANDOM_STATE}.csv')
print('🎯 Índices amostra aleatórios (teste 10%):', indices_aleatorios)


In [ ]:
# ==============================================
# Salvar X_test_basic_full.csv e X_train_basic_full.csv
# (aderente ao 8/1/1; "train" = 90% se X_90 existir, senão 80%)
# com todas as colunas do arquivo cru + y, y_pred, y_proba
# ==============================================

import pandas as pd
import numpy as np

# ---------------- Carregar o arquivo cru ----------------
raw_df = pd.read_csv(DATASET_PATH)

# Para FHS, a coluna alvo é 'TenYearCHD' (0/1); se existir, manter. Caso tenha outro nome, ajuste TARGET_COLUMN anteriormente
if 'TenYearCHD' in raw_df.columns:
    raw_df['y'] = raw_df['TenYearCHD']
elif 'y' not in raw_df.columns:
    raise ValueError("❌ Nenhuma coluna alvo encontrada no dataset (esperado 'TenYearCHD' ou 'y').")

# ---------------- Utilitários ----------------
def _safe_proba(y_pred):
    """Gera probabilidade binária simples (1.0 se 1, 0.0 se 0) para fallback."""
    return np.where(y_pred == 1, 1.0, 0.0)

def _pick_model():
    """Escolhe o modelo disponível com esta prioridade."""
    for mname in ['model_basic', 'final_model', 'model']:
        if mname in globals():
            return globals()[mname]
    raise RuntimeError("❌ Nenhum modelo encontrado (esperado 'model_basic' ou 'final_model' ou 'model').")

def _predict_with_fallback(model, X, y_pred_bin_existing=None):
    """Retorna (y_pred_bin, y_proba) usando predict_proba quando possível."""
    try:
        y_proba = model.predict_proba(X)[:, 1]
        y_pred_bin = (y_proba >= 0.5).astype(int) if y_pred_bin_existing is None else y_pred_bin_existing
    except Exception:
        # Sem predict_proba: tenta usar y_pred_bin existente ou gera
        if y_pred_bin_existing is None:
            try:
                y_pred_bin = model.predict(X)
            except Exception:
                raise RuntimeError("❌ Não foi possível obter predições para gerar y_proba/y_pred.")
        else:
            y_pred_bin = y_pred_bin_existing
        y_proba = _safe_proba(y_pred_bin)
    return y_pred_bin, y_proba

def save_full_split(indices, y_true, y_pred, y_proba, split_name):
    """Monta um DataFrame completo com o conjunto original + y_true, y_pred, y_proba e salva."""
    df_full = raw_df.loc[indices].copy()
    df_full['y'] = list(pd.Series(y_true, index=indices).values)
    df_full['y_pred'] = list(pd.Series(y_pred, index=indices).values)
    df_full['y_proba'] = list(pd.Series(y_proba, index=indices).values)
    out_path = BASICO_CSV / f'X_{split_name}_basic_full.csv'
    df_full.to_csv(out_path, index=True)
    print(f"✅ Arquivo salvo: {out_path} (linhas: {len(df_full)})")

# ---------------- Preparar “train” (90% se existir X_90) e “test” (10%) ----------------
# Preferir 90% (train+val) se já montado no pipeline 8/1/1
if 'X_90' in globals() and 'y_90' in globals():
    X_train_like = X_90
    y_train_like = y_90
else:
    # fallback para 80% caso ainda não tenha concatenado validação
    if 'X_train' not in globals() or 'y_train' not in globals():
        raise RuntimeError("❌ Splits não encontrados (esperado X_90/y_90 ou X_train/y_train e X_test/y_test).")
    X_train_like = X_train
    y_train_like = y_train

if 'X_test' not in globals() or 'y_test' not in globals():
    raise RuntimeError("❌ Split de teste não encontrado (esperado X_test e y_test).")

# ---------------- Obter predições com o modelo disponível ----------------
_model = _pick_model()

# Para TESTE (10%)
try:
    y_pred_test_bin  # pode já existir
    y_proba_test     # pode já existir
except NameError:
    y_pred_test_bin = None
    y_proba_test = None

if y_proba_test is None:
    y_pred_test_bin, y_proba_test = _predict_with_fallback(_model, X_test, y_pred_test_bin)

# Para TREINO (90% preferencialmente; senão 80%)
try:
    y_pred_train_bin   # pode já existir
    y_proba_train      # pode já existir
except NameError:
    y_pred_train_bin = None
    y_proba_train = None

if y_proba_train is None:
    y_pred_train_bin, y_proba_train = _predict_with_fallback(_model, X_train_like, y_pred_train_bin)

# ---------------- Salvamentos (mantendo os mesmos nomes de arquivos) ----------------
save_full_split(X_test.index,        y_test,        y_pred_test_bin,  y_proba_test,  'test')
save_full_split(X_train_like.index,  y_train_like,  y_pred_train_bin, y_proba_train, 'train')
